In [ ]:
import pandas as pd
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (KFold, StratifiedKFold)
from sklearn.model_selection import train_test_split
from sklearn.model_selection import (
    RandomizedSearchCV,
    GridSearchCV,
)
from sklearn.metrics import make_scorer

from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored

df = pd.read_csv("ml.csv")

In [ ]:
survival_df = df.copy()

survival_df["service_date"] = pd.to_datetime(
    survival_df["service_date"]
)

survival_df = survival_df.sort_values(
    ["vehicle_id", "service_date"]
).reset_index(drop=True)

In [ ]:
survival_records = []

for vehicle_id, vehicle_data in survival_df.groupby("vehicle_id"):

    vehicle_data = (
        vehicle_data
        .sort_values("service_date")
        .reset_index(drop=True)
    )

    # --------------------------------------------------
    # Find first failure
    # --------------------------------------------------

    failure_rows = vehicle_data[
        vehicle_data["failure_occurred"] == 1
    ]

    if not failure_rows.empty:

        # First failure
        first_failure_idx = failure_rows.index[0]

        failure_date = vehicle_data.loc[
            first_failure_idx,
            "service_date"
        ]

        # --------------------------------------------------
        # Every observation BEFORE the first failure
        # becomes a prediction point
        # --------------------------------------------------

        prediction_rows = vehicle_data.iloc[:first_failure_idx]

        for _, prediction_row in prediction_rows.iterrows():

            observation_date = prediction_row["service_date"]

            duration_days = (
                failure_date - observation_date
            ).days

            # Skip invalid/zero durations
            if duration_days <= 0:
                continue

            survival_records.append({

                "vehicle_id": vehicle_id,

                # ------------------------------------------
                # Static vehicle information
                # ------------------------------------------

                "model": prediction_row["model"],
                "engine_type": prediction_row["engine_type"],
                "engine_cc": prediction_row["engine_cc"],
                "weight_kg": prediction_row["weight_kg"],
                "vehicle_class": prediction_row["vehicle_class"],
                "drivetrain": prediction_row["drivetrain"],

                # ------------------------------------------
                # Vehicle state AT prediction time
                # ------------------------------------------

                "vehicle_age": prediction_row["vehicle_age"],
                "condition": prediction_row["condition"],
                "mileage": prediction_row["mileage"],
                "missed_services": prediction_row["missed_services"],

                # ------------------------------------------
                # Environment / usage AT prediction time
                # ------------------------------------------

                "season": prediction_row["season"],
                "ambient_temp": prediction_row["ambient_temp"],
                "service_type": prediction_row["service_type"],

                # ------------------------------------------
                # Survival information
                # ------------------------------------------

                "observation_date": observation_date,
                "failure_date": failure_date,
                "duration_days": duration_days,
                "event": 1
            })

    else:

        # --------------------------------------------------
        # No failure → right censored
        # --------------------------------------------------

        # Use the last available observation as the censoring
        # endpoint.
        #
        # We can also create prediction points for earlier
        # observations, but need a common observation end date.
        # --------------------------------------------------

        observation_end_date = vehicle_data["service_date"].max()

        for _, prediction_row in vehicle_data.iloc[:-1].iterrows():

            observation_date = prediction_row["service_date"]

            duration_days = (
                observation_end_date - observation_date
            ).days

            if duration_days <= 0:
                continue

            survival_records.append({

                "vehicle_id": vehicle_id,

                # ------------------------------------------
                # Static vehicle information
                # ------------------------------------------

                "model": prediction_row["model"],
                "engine_type": prediction_row["engine_type"],
                "engine_cc": prediction_row["engine_cc"],
                "weight_kg": prediction_row["weight_kg"],
                "vehicle_class": prediction_row["vehicle_class"],
                "drivetrain": prediction_row["drivetrain"],

                # ------------------------------------------
                # Vehicle state AT prediction time
                # ------------------------------------------

                "vehicle_age": prediction_row["vehicle_age"],
                "condition": prediction_row["condition"],
                "mileage": prediction_row["mileage"],
                "missed_services": prediction_row["missed_services"],

                # ------------------------------------------
                # Environment / usage AT prediction time
                # ------------------------------------------

                "season": prediction_row["season"],
                "ambient_temp": prediction_row["ambient_temp"],
                "service_type": prediction_row["service_type"],

                # ------------------------------------------
                # Survival information
                # ------------------------------------------

                "observation_date": observation_date,
                "failure_date": pd.NaT,
                "duration_days": duration_days,
                "event": 0
            })


survival_data = pd.DataFrame(survival_records)

In [ ]:
print("Survival observations:", len(survival_data))

print("\nEvents:")
print(
    survival_data["event"]
    .value_counts()
    .rename({
        0: "Censored",
        1: "Failure"
    })
)

print("\nDuration:")
print(
    survival_data["duration_days"]
    .describe()
)

In [ ]:
survival_data = (
    survival_data
    .sort_values("duration_days", ascending=False)
    .drop_duplicates(subset="vehicle_id", keep="first")
    .reset_index(drop=True)
)
print("Rows after deduplication:", len(survival_data))
print("Unique vehicles:", survival_data["vehicle_id"].nunique())
display(survival_data[survival_data['vehicle_id'] == 'V2500'])

In [ ]:
expected_ids = {f"V{i}" for i in range(1, 10001)}
actual_ids = set(survival_data["vehicle_id"])

missing_ids = sorted(
    expected_ids - actual_ids,
    key=lambda x: int(x[1:])
)

print(f"Expected vehicles: {len(expected_ids)}")
print(f"Found vehicles:    {len(expected_ids & actual_ids)}")
print(f"Missing vehicles:  {len(missing_ids)}")

print("\nMissing vehicle IDs:")
print(missing_ids)

In [ ]:
survival_data[survival_data["event"] == 1]

In [ ]:
# ============================================================
# 1. Load data
# ============================================================

survival_features = [
    "model",
    "engine_type",
    "engine_cc",
    "weight_kg",
    "vehicle_class",
    "drivetrain",
    "vehicle_age",
    # "condition",
    "mileage",
    "missed_services",
    "season",
    "ambient_temp",
    "service_type",
]

survival_target = [
    "duration_days",
    "event"
]


# ============================================================
# 2. Select features and target
# ============================================================

X = survival_data[survival_features].copy()

# event must be boolean
y = Surv.from_dataframe(
    event="event",
    time="duration_days",
    data=survival_data[survival_target]
)


# ============================================================
# 3. Identify feature types
# ============================================================

categorical_features = [
    "model",
    "engine_type",
    "vehicle_class",
    "drivetrain",
    "service_type",
    "season",
]

numerical_features = [
    "engine_cc",
    "weight_kg",
    "vehicle_age",
    "mileage",
    "missed_services",
    # "condition",
    "ambient_temp"
]


# ============================================================
# 4. Preprocessing
# ============================================================

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])


# ============================================================
# 5. train/test split
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

train_idx = X_train.index
test_idx = X_test.index

# ============================================================
# 6. Verify vehicle separation
# ============================================================

train_vehicle_ids = set(
    survival_data.iloc[train_idx]["vehicle_id"]
)

test_vehicle_ids = set(
    survival_data.iloc[test_idx]["vehicle_id"]
)

print(f"Training vehicles: {len(train_vehicle_ids)}")
print(f"Testing vehicles:  {len(test_vehicle_ids)}")


# ============================================================
# 7. Build full preprocessing + RSF pipeline
# ============================================================

rsf_pipeline = Pipeline([
    ("preprocessor", preprocessor),

    ("rsf", RandomSurvivalForest(
        n_estimators=100,
        min_samples_split=30,
        min_samples_leaf=100,
        max_features="sqrt",
        max_depth=5,
        n_jobs=-1,
        random_state=42
    ))
])


# ============================================================
# 8. Train full pipeline
# ============================================================

rsf_pipeline.fit(
    X_train,
    y_train
)

print("RSF pipeline trained successfully.")

# ============================================================
# 9. Evaluate
# ============================================================

risk_scores = rsf_pipeline.predict(X_test)

c_index = concordance_index_censored(
    y_test["event"],
    y_test["duration_days"],
    risk_scores
)[0]

print(f"Concordance Index: {c_index:.4f}")

In [ ]:
# ============================================================
# 10. Save
# ============================================================

import joblib

joblib.dump(
    rsf_pipeline,
    "rsf_pipeline_simple.joblib"
)

print("Full RSF pipeline saved.")

In [ ]:
import joblib

# Load pipeline
rsf_pipeline = joblib.load("rsf_pipeline_simple.joblib")

# Predict on raw test data
risk_scores = rsf_pipeline.predict(X_test)

# Evaluate
c_index = concordance_index_censored(
    y_test["event"],
    y_test["duration_days"],
    risk_scores
)[0]

print(f"Test C-index: {c_index:.4f}")

In [ ]:
# ============================================================
# Hyperparameter tuning pipeline
# ============================================================

rsf_pipeline = Pipeline([
    ("preprocessor", preprocessor),

    ("rsf", RandomSurvivalForest(
        random_state=42,
        n_jobs=-1
    ))
])

def cindex_scorer(estimator, X, y):
    risk_scores = estimator.predict(X)

    return concordance_index_censored(
        y["event"],
        y["duration_days"],
        risk_scores
    )[0]

# ============================================================
# Grouped cross-validation
# ============================================================

cv = KFold(
    n_splits=3,
)

# ============================================================
# Hyperparameter search space
# ============================================================

param_distributions = {
    "rsf__n_estimators": [
        100,
        200,
        300,
    ],

    "rsf__min_samples_split": [
        10,
        30,
        50,
    ],

    "rsf__min_samples_leaf": [
        5,
        50,
        100,
    ],

    "rsf__max_features": [
        "sqrt",
    ],

    "rsf__max_depth": [
        5,
        10,
        15,
    ]
}



In [ ]:
# search = RandomizedSearchCV(
#     estimator=rsf_pipeline,

#     param_distributions=param_distributions,

#     n_iter=20,

#     scoring=cindex_scorer,

#     cv=cv,

#     n_jobs=1,

#     verbose=3,

#     random_state=42,

#     refit=True,

#     return_train_score=True
# )

In [ ]:
search = GridSearchCV(
    estimator=rsf_pipeline,
    param_grid=param_distributions, 
    scoring=cindex_scorer,
    cv=cv,
    n_jobs=1,
    verbose=3,
    refit=True,
    return_train_score=True
)

In [ ]:
# ============================================================
# Run hyperparameter tuning
# ============================================================

search.fit(
    X_train,
    y_train,
)

print("\nBest parameters:")
print(search.best_params_)

print(
    f"\nBest CV C-index: "
    f"{search.best_score_:.4f}"
)